# Partie 1: Connexion & Préparation des Données 

# la connexion

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv
import streamlit as st
load_dotenv()

engine_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(engine_url)

In [2]:
df = pd.read_csv("../data/financecore_clean.csv")
df.columns

Index(['transaction_id', 'client_id', 'date_transaction', 'montant', 'devise',
       'taux_change_eur', 'montant_eur', 'categorie', 'produit', 'agence',
       'type_operation', 'statut', 'score_credit_client', 'segment_client',
       'solde_avant', 'is_anomalie', 'annee', 'mois', 'trimestre',
       'semaine-jour', 'montant_eur_verifie', 'categorie_risque',
       'total_credit', 'total_debit', 'solde_net', 'nb_transaction',
       'montant_moyen', 'nb_produit', 'taux_rejet'],
      dtype='str')

# preparation des données

In [3]:
load_agences= pd.read_sql("SELECT * FROM agence", engine)
load_produits= pd.read_sql("SELECT * FROM produit", engine)
load_categories = pd.read_sql("SELECT * FROM categorie", engine)
load_clients = pd.read_sql("SELECT * FROM client", engine)
df_comptes= pd.read_sql("SELECT * FROM compte", engine)
transactions = pd.read_sql("SELECT * FROM transaction", engine)

In [4]:
print(load_categories.dtypes)
print(load_agences.dtypes)
print(load_categories.dtypes)
print(load_clients.dtypes)
print(df_comptes.dtypes)
print(transactions.dtypes)

categorie_id     int64
nom_categorie      str
dtype: object
agence_id     int64
nom_agence      str
dtype: object
categorie_id     int64
nom_categorie      str
dtype: object
client_id                  str
score_credit_client    float64
nom_segment                str
dtype: object
compte_id           int64
client_id             str
agence_id           int64
produit_id          int64
categorie_risque      str
dtype: object
transaction_id                 str
compte_id                    int64
categorie_id                 int64
date_transaction    datetime64[us]
montant                    float64
montant_eur                float64
code_devise                    str
type_operation                 str
statut                         str
solde_avant                float64
is_anomalie                   bool
dtype: object


In [10]:
print(load_agences.columns)
print(load_produits.columns)
print(load_categories.columns)
print(load_clients.columns)
print(df_comptes.columns)
print(transactions .columns)

Index(['agence_id', 'nom_agence'], dtype='str')
Index(['produit_id', 'nom_produit'], dtype='str')
Index(['categorie_id', 'nom_categorie'], dtype='str')
Index(['client_id', 'score_credit_client', 'nom_segment'], dtype='str')
Index(['compte_id', 'client_id', 'agence_id', 'produit_id',
       'categorie_risque'],
      dtype='str')
Index(['transaction_id', 'compte_id', 'categorie_id', 'date_transaction',
       'montant', 'montant_eur', 'code_devise', 'type_operation', 'statut',
       'solde_avant', 'is_anomalie'],
      dtype='str')


In [6]:
def kpis_query():
    return """
    SELECT 
        COUNT(*) AS total_transactions,
        SUM(montant) AS total_ca,
        COUNT(DISTINCT client_id) AS clients_actifs,
        AVG(montant) AS marge_moyenne
    FROM transaction
    """

# partie 2:  Page 1 — Vue Exécutive

#KPIs en cartes : Volume total transactions, CA total, Nombre de clients actifs, Marge moyenne

#Graphique lignes : évolution mensuelle des débits et crédits (2022–2024)

#Graphique barres : CA par agence et par produit bancaire

#Diagramme circulaire : répartition des clients par segment (Premium / Standard / Risqué)

# partie 3 : Page 2 — Analyse des Risques

#Heatmap : corrélation entre score crédit, montant et taux de rejet

#Scatter plot : score crédit vs montant transaction, coloré par catégorie de risque

#Tableau des 10 clients les plus à risque avec indicateurs visuels colorés

# partie 4:  Filtres Interactifs

#Sidebar avec filtres : Agence, Segment client, Produit, Période (slider années)

#Tous les graphiques réactifs aux filtres sélectionnés

#Bouton d'export CSV des données filtrées